# Scanine: subject check evaluation (step 1)

Measures whether a ready-made object detector (**EfficientDet-Lite0**, COCO classes) can tell
"this photo shows a dog" from "this photo shows something else" **before** a photo reaches the
Scanine classifier.

**This notebook does not touch the Scanine model.** It never loads `scanine.tflite` or the
`.keras` file, never retrains anything, and only *reads* your dataset photos.
It writes only to:
- `/content/subject_eval/` (scratch space on the Colab machine, gone when the session ends)
- a new timestamped folder under `MyDrive/subject_check_eval/` (the results)

**How to run:** check the settings in the next cell, then *Runtime → Run all*.
At the end, send back the printed **RESULT** block and the files in the results folder.

**Pass mark:** at most 2% of real dog photos wrongly rejected, while catching as many
not-dog photos as possible.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
# The model run whose val + test photos form the "dog" set.
RUN_NAME = "20261029_1757_mnv3s_baseline_seed42"

# Path to that run's split.csv in Drive. Leave empty to search MyDrive for
# <RUN_NAME>/split.csv (can take a few minutes on a big Drive).
SPLIT_CSV = ""

# Optional: a Drive folder that holds the dataset photos (searched by file name).
# Leave empty to download each photo by the Drive ID stored in split.csv.
DOG_IMAGE_DIR = ""

# Your own not-dog photos (human skin, blankets, floors, cats...). Optional but
# the most realistic test. The folder is created if it doesn't exist.
OWN_NEGATIVES_DIR = "/content/drive/MyDrive/subject_check_eval/my_not_dog_photos"

# How many public COCO photos (no dog in them) to add as not-dog examples.
COCO_NEGATIVES = 150

# Also add close-up textures (fabric, matted, wrinkled...) from the DTD dataset.
# Hard negatives, but a ~600 MB download. Off by default.
USE_DTD_TEXTURES = False

# Pass mark: highest acceptable share of real dog photos rejected.
MAX_FALSE_REJECT = 0.02

# The exact detector file that would ship in the app.
DETECTOR_URL = ("https://storage.googleapis.com/mediapipe-models/object_detector/"
                "efficientdet_lite0/int8/latest/efficientdet_lite0.tflite")

OUT_ROOT = "/content/drive/MyDrive/subject_check_eval"
SEED = 42

In [ ]:
# ── Setup ───────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ai-edge-litert'], check=False)

import os, io, json, glob, random, shutil, zipfile, time, urllib.request
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

random.seed(SEED)
np.random.seed(SEED)

WORK = '/content/subject_eval'          # scratch on the Colab machine
os.makedirs(WORK, exist_ok=True)
OUT_DIR = os.path.join(OUT_ROOT, time.strftime('%Y%m%d_%H%M'))
os.makedirs(OUT_DIR, exist_ok=True)
print('Results will be saved to:', OUT_DIR)

## 1. Dog photos: the run's val + test split

In [ ]:
if not SPLIT_CSV:
    print(f'Searching MyDrive for {RUN_NAME}/split.csv ...')
    hits = glob.glob(f'/content/drive/MyDrive/**/{RUN_NAME}/split.csv', recursive=True)
    if not hits:
        raise FileNotFoundError(
            f'No {RUN_NAME}/split.csv found in MyDrive. Set SPLIT_CSV in the settings cell.')
    SPLIT_CSV = hits[0]
print('split.csv:', SPLIT_CSV)

split = pd.read_csv(SPLIT_CSV)
dogs = split[split['split'].isin(['val', 'test'])
             & (split['excluded'].astype(str) != 'True')].copy()
print(len(dogs), 'dog photos')
print(dogs.groupby(['label', 'split']).size().unstack(fill_value=0))

In [ ]:
DOG_DIR = os.path.join(WORK, 'dogs')
os.makedirs(DOG_DIR, exist_ok=True)

def local_path(row):
    return os.path.join(DOG_DIR, f"{row['label']}__{row['name']}")

if DOG_IMAGE_DIR:
    index = defaultdict(list)
    for p in glob.glob(os.path.join(DOG_IMAGE_DIR, '**', '*'), recursive=True):
        if os.path.isfile(p):
            index[os.path.basename(p).lower()].append(p)

    def get_photo(row, dest):
        n = row['name'].lower()
        found = index.get(n) or index.get(n + '.jpg') or []
        # Same file name in two class folders: prefer the one under its label.
        same_label = [p for p in found if row['label'].lower() in p.lower()]
        src = (same_label or found or [None])[0]
        if src is None:
            return False
        shutil.copyfile(src, dest)   # copy out of Drive; the original is untouched
        return True
else:
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    service = build('drive', 'v3')

    def get_photo(row, dest):
        try:
            req = service.files().get_media(fileId=row['drive_id'])
            with open(dest, 'wb') as f:
                dl = MediaIoBaseDownload(f, req)
                done = False
                while not done:
                    _, done = dl.next_chunk()
            return True
        except Exception as e:
            if os.path.exists(dest):
                os.remove(dest)
            print('  could not get', row['name'], '-', e)
            return False

paths, missing = [], []
for i, (_, row) in enumerate(dogs.iterrows()):
    dest = local_path(row)
    ok = os.path.exists(dest) or get_photo(row, dest)
    paths.append(dest if ok else None)
    if not ok:
        missing.append(row['name'])
    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(dogs)}')
dogs['path'] = paths
print(f'Got {dogs.path.notna().sum()} of {len(dogs)} dog photos; missing {len(missing)}')
dogs = dogs[dogs.path.notna()]

## 2. Not-dog photos

In [ ]:
NEG_DIR = os.path.join(WORK, 'not_dogs')
os.makedirs(NEG_DIR, exist_ok=True)
neg_rows = []

# a) COCO val2017 photos that contain no dog, grouped by what they do show.
NEG_GROUPS = {
    'cat': ['cat'],
    'person': ['person'],
    'other_animal': ['bird', 'horse', 'sheep', 'cow', 'elephant', 'bear',
                     'zebra', 'giraffe', 'teddy bear'],
    'indoor': ['couch', 'bed', 'tv', 'laptop', 'cell phone', 'chair',
               'dining table', 'potted plant'],
}
if COCO_NEGATIVES > 0:
    ann_zip = os.path.join(WORK, 'coco_annotations.zip')
    if not os.path.exists(ann_zip):
        print('Downloading COCO annotations (~240 MB) ...')
        urllib.request.urlretrieve(
            'http://images.cocodataset.org/annotations/annotations_trainval2017.zip', ann_zip)
    with zipfile.ZipFile(ann_zip) as z:
        coco = json.loads(z.read('annotations/instances_val2017.json'))
    cat_name = {c['id']: c['name'] for c in coco['categories']}
    images = {im['id']: im for im in coco['images']}
    by_image = defaultdict(list)
    for a in coco['annotations']:
        by_image[a['image_id']].append(a)

    # Keep photos where the group's object is large (closer to "fills the frame").
    candidates, used = defaultdict(list), set()
    for iid, anns in sorted(by_image.items()):
        if any(cat_name[a['category_id']] == 'dog' for a in anns):
            continue
        im = images[iid]
        frame = im['width'] * im['height']
        for group, members in NEG_GROUPS.items():
            biggest = max((a['area'] / frame for a in anns
                           if cat_name[a['category_id']] in members), default=0)
            if biggest >= 0.15 and iid not in used:
                candidates[group].append(iid)
                used.add(iid)

    per_group = COCO_NEGATIVES // len(NEG_GROUPS)
    for group, ids in candidates.items():
        for iid in random.sample(ids, min(per_group, len(ids))):
            fn = images[iid]['file_name']
            dest = os.path.join(NEG_DIR, f'coco_{group}_{fn}')
            if not os.path.exists(dest):
                urllib.request.urlretrieve(f'http://images.cocodataset.org/val2017/{fn}', dest)
            neg_rows.append({'group': f'coco_{group}', 'path': dest})
    print('COCO not-dog photos:', sum(r['group'].startswith('coco') for r in neg_rows))

# b) Close-up textures (optional).
if USE_DTD_TEXTURES:
    import tensorflow_datasets as tfds
    ds, info = tfds.load('dtd', split='test', with_info=True)
    names = info.features['label'].names
    wanted = {'fibrous', 'matted', 'knitted', 'woven', 'wrinkled', 'blotchy', 'stained'}
    taken = defaultdict(int)
    for ex in tfds.as_numpy(ds):
        name = names[int(ex['label'])]
        if name in wanted and taken[name] < 8:
            dest = os.path.join(NEG_DIR, f'dtd_{name}_{taken[name]}.jpg')
            Image.fromarray(ex['image']).save(dest, quality=92)
            neg_rows.append({'group': 'dtd_texture', 'path': dest})
            taken[name] += 1
    print('Texture photos:', sum(taken.values()))

# c) Your own photos.
os.makedirs(OWN_NEGATIVES_DIR, exist_ok=True)
own = [p for p in glob.glob(os.path.join(OWN_NEGATIVES_DIR, '**', '*'), recursive=True)
       if p.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.heic'))]
for p in own:
    neg_rows.append({'group': 'own', 'path': p})
print(f'Your own not-dog photos: {len(own)} (folder: {OWN_NEGATIVES_DIR})')

negs = pd.DataFrame(neg_rows)
print(negs.group.value_counts())

## 3. Run the detector

In [ ]:
DET_PATH = os.path.join(WORK, os.path.basename(DETECTOR_URL))
if not os.path.exists(DET_PATH):
    urllib.request.urlretrieve(DETECTOR_URL, DET_PATH)
print('Detector:', DET_PATH, f'{os.path.getsize(DET_PATH) / 1e6:.1f} MB')

# A .tflite with metadata is also a zip that carries its label map.
labels = None
try:
    with zipfile.ZipFile(DET_PATH) as z:
        txt = [n for n in z.namelist() if n.endswith('.txt')]
        if txt:
            labels = [l.strip() for l in z.read(txt[0]).decode().splitlines()]
except zipfile.BadZipFile:
    pass
if not labels:
    raise RuntimeError('No label map inside the detector file; cannot name its classes.')
print(len(labels), 'labels, e.g.', labels[:5])

interp = Interpreter(model_path=DET_PATH, num_threads=4)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
IN_H, IN_W = int(inp['shape'][1]), int(inp['shape'][2])
outs = interp.get_output_details()
print('Input', inp['shape'], inp['dtype'])

def _whole(a):
    return np.allclose(a, np.round(a))

def detect(path, top=10):
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    x = np.asarray(im.resize((IN_W, IN_H), Image.BILINEAR))[None]
    if inp['dtype'] == np.uint8:
        x = x.astype(np.uint8)
    elif inp['dtype'] == np.int8:
        x = (x.astype(np.int16) - 128).astype(np.int8)
    else:
        x = (x.astype(np.float32) - 127.5) / 127.5
    interp.set_tensor(inp['index'], x)
    interp.invoke()
    arrs = [interp.get_tensor(o['index']) for o in outs]
    boxes = next(a[0] for a in arrs if a.ndim == 3 and a.shape[-1] == 4)
    vecs = [a[0] for a in arrs if a.ndim == 2]
    # Class ids are whole numbers; scores are not.
    classes, scores = ((vecs[0], vecs[1]) if _whole(vecs[0]) and not _whole(vecs[1])
                       else (vecs[1], vecs[0]))
    dets = []
    for b, c, s in zip(boxes, classes, scores):
        c = int(round(c))
        name = labels[c] if 0 <= c < len(labels) else f'class_{c}'
        ymin, xmin, ymax, xmax = np.clip(b, 0, 1)
        dets.append((name, float(s), float((ymax - ymin) * (xmax - xmin))))
    dets.sort(key=lambda d: -d[1])
    return dets[:top]

# Sanity check: the detector should name COCO cats as cats. If not, the
# label mapping above is off and the results below are not trustworthy.
cat_paths = negs[negs.group == 'coco_cat'].path.tolist()[:20]
if cat_paths:
    hit = np.mean([detect(p)[0][0] == 'cat' for p in cat_paths])
    print(f'Sanity check: top label is "cat" on {hit:.0%} of COCO cat photos')
    if hit < 0.5:
        print('WARNING: label mapping looks wrong; stop and send this output back.')

In [ ]:
# "Strong" non-dog classes: things that are clearly not a dog's skin.
STRONG = {'person', 'cat', 'tv', 'laptop', 'cell phone', 'bed', 'couch', 'chair',
          'dining table', 'book', 'keyboard', 'remote', 'toilet', 'sink', 'car', 'bird'}

def summarise(dets):
    dog = max((s for n, s, _ in dets if n == 'dog'), default=0.0)
    others = [(n, s, a) for n, s, a in dets if n != 'dog']
    strong = [(n, s, a) for n, s, a in others if n in STRONG]
    top = others[0] if others else ('', 0.0, 0.0)
    top_strong = strong[0] if strong else ('', 0.0, 0.0)
    return {
        'dog_score': dog,
        'other_label': top[0], 'other_score': top[1], 'other_area': top[2],
        'strong_label': top_strong[0], 'strong_score': top_strong[1],
        'strong_area': top_strong[2],
        'detections': json.dumps([(n, round(s, 3), round(a, 3)) for n, s, a in dets[:5]]),
    }

records = []
t0 = time.time()
for _, r in dogs.iterrows():
    records.append({'set': 'dog', 'group': r['label'], 'path': r['path'], **summarise(detect(r['path']))})
for _, r in negs.iterrows():
    records.append({'set': 'not_dog', 'group': r['group'], 'path': r['path'], **summarise(detect(r['path']))})
res = pd.DataFrame(records)
print(f'{len(res)} photos in {time.time() - t0:.0f}s')
res.to_csv(os.path.join(OUT_DIR, 'detections.csv'), index=False)

print('\nHow often the detector sees a dog (score >= 0.3) in the dog photos, by class:')
print(res[res['set'] == 'dog'].groupby('group').dog_score.apply(lambda s: f'{(s >= 0.3).mean():.0%}'))

## 4. Find thresholds

A photo is **rejected** ("not a dog") when the detector
- does **not** see a dog (dog score below `keep`), **and**
- **does** see something else with score at least `other`, covering at least `min_area` of the frame.

Everything else passes: either *dog* (dog score ≥ `keep`) or *unsure* (nothing confident found,
typical for close-ups). Two variants are tried: counting **any** non-dog class, or only the
**strong** list (person, cat, screens, furniture...).

In [ ]:
def wilson_upper(k, n, z=1.96):
    if n == 0:
        return float('nan')
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    m = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c + m) / d

def rejected(df, variant, keep, other, min_area):
    pre = 'other' if variant == 'any' else 'strong'
    return ((df.dog_score < keep) & (df[f'{pre}_score'] >= other)
            & (df[f'{pre}_area'] >= min_area))

dog_df, neg_df = res[res['set'] == 'dog'], res[res['set'] == 'not_dog']
rows = []
for variant in ['any', 'strong']:
    for keep in [0.2, 0.3, 0.4, 0.5]:
        for other in np.round(np.arange(0.30, 0.86, 0.05), 2):
            for min_area in [0.0, 0.1, 0.25]:
                fr = rejected(dog_df, variant, keep, other, min_area)
                cr = rejected(neg_df, variant, keep, other, min_area)
                rows.append({
                    'variant': variant, 'keep': keep, 'other': other, 'min_area': min_area,
                    'dog_rejected': fr.mean(), 'dog_rejected_upper95': wilson_upper(fr.sum(), len(fr)),
                    'not_dog_caught': cr.mean(),
                    'dog_unsure': ((dog_df.dog_score < keep) & ~fr).mean(),
                })
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT_DIR, 'threshold_table.csv'), index=False)

ok = table[table.dog_rejected <= MAX_FALSE_REJECT]
ok = ok.sort_values(['not_dog_caught', 'dog_rejected', 'other'], ascending=[False, True, False])
pd.set_option('display.float_format', '{:.3f}'.format)
print(f'Settings that pass the mark (dog photos rejected <= {MAX_FALSE_REJECT:.0%}): {len(ok)}')
print(ok.head(10).to_string(index=False))

In [ ]:
print('=' * 70)
print('RESULT  (send this block back)')
print('=' * 70)
print('Detector:', os.path.basename(DETECTOR_URL), '| dog photos:', len(dog_df),
      '| not-dog photos:', len(neg_df), {str(k): int(v) for k, v in neg_df.group.value_counts().items()})
if ok.empty:
    best = None
    closest = table.sort_values('dog_rejected').iloc[0]
    print(f'NO setting meets the {MAX_FALSE_REJECT:.0%} mark. Lowest dog rejection is '
          f"{closest.dog_rejected:.1%} (catching {closest.not_dog_caught:.0%} of not-dog photos).")
    print('EfficientDet alone is not enough -> next step: test ML Kit on the phone.')
else:
    best = ok.iloc[0]
    fr = rejected(dog_df, best.variant, best.keep, best.other, best.min_area)
    cr = rejected(neg_df, best.variant, best.keep, best.other, best.min_area)
    print(f"Best setting: variant={best.variant} keep={best.keep} other={best.other} "
          f"min_area={best.min_area}")
    print(f"  dog photos wrongly rejected: {fr.sum()}/{len(fr)} = {fr.mean():.1%} "
          f"(95% upper bound {best.dog_rejected_upper95:.1%})")
    print(f"  not-dog photos caught:       {cr.sum()}/{len(cr)} = {cr.mean():.1%}")
    print(f"  dog photos marked unsure:    {best.dog_unsure:.1%}")
    print('  rejected by class:', {str(k): int(v) for k, v in dog_df[fr].group.value_counts().items()})
    print('  caught by group:  ', {str(g): f'{cr[neg_df.group == g].mean():.0%}'
                                   for g in sorted(neg_df.group.unique())})
    with open(os.path.join(OUT_DIR, 'best_setting.json'), 'w') as f:
        json.dump({'detector': os.path.basename(DETECTOR_URL),
                   'variant': best.variant, 'keep': float(best.keep),
                   'other': float(best.other), 'min_area': float(best.min_area),
                   'strong_classes': sorted(STRONG)}, f, indent=2)
print('Files saved in', OUT_DIR)

## 5. Look at the mistakes

In [ ]:
def gallery(df, title, fname, n=24):
    df = df.head(n)
    if df.empty:
        print(f'{title}: none')
        return
    cols = 6
    rows_ = (len(df) + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * 2.6, rows_ * 3.0))
    for ax in np.ravel(axes):
        ax.axis('off')
    for ax, (_, r) in zip(np.ravel(axes), df.iterrows()):
        im = ImageOps.exif_transpose(Image.open(r.path)).convert('RGB')
        im.thumbnail((300, 300))
        ax.imshow(im)
        ax.set_title(f"{r.group}\ndog {r.dog_score:.2f} | {r.other_label} {r.other_score:.2f}",
                     fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=110)
    plt.show()

if best is not None:
    fr = rejected(dog_df, best.variant, best.keep, best.other, best.min_area)
    cr = rejected(neg_df, best.variant, best.keep, best.other, best.min_area)
    gallery(dog_df[fr], 'Dog photos wrongly rejected', 'wrongly_rejected_dogs.png')
    gallery(neg_df[~cr].sort_values('other_score'), 'Not-dog photos that got through',
            'missed_not_dogs.png')
else:
    gallery(dog_df.sort_values('dog_score'), 'Dog photos the detector is least sure about',
            'least_sure_dogs.png')